In [37]:
pip install jieba

Note: you may need to restart the kernel to use updated packages.


In [38]:
import os, types, io
import pandas as pd
from botocore.client import Config
import ibm_boto3

# Shim so pandas can iterate over the response body
def __iter__(self):
    return iter(self.read().splitlines())

# Create the COS client using the public S3 endpoint for eu‑de
cos_client = ibm_boto3.client(
    service_name='s3',
    ibm_api_key_id='0x5HjUXotavicsDWNRN7kPT6gryrLiE-mt__PG8Vg7jI',            # your API key
    ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.eu-de.cloud-object-storage.appdomain.cloud'       # public endpoint
)

bucket = 'uploadtrialsentimenthuan-donotdelete-pr-cxwn3ymcknpdd3'
object_key = 'wiki_extracts_with_labels.csv'

# Stream the object from COS
body = cos_client.get_object(Bucket=bucket, Key=object_key)['Body']

# Attach our iterator shim if needed
if not hasattr(body, "__iter__"):
    body.__iter__ = types.MethodType(__iter__, body)

# Wrap in a text buffer and specify encoding to avoid UTF-8 errors
text_stream = io.TextIOWrapper(body,  encoding='utf-8')

# Read into DataFrame
df_articles = pd.read_csv(text_stream, encoding='utf-8')

# Show the first 10 rows
df_articles.head(10)

,qid,language,title,extract,semantic_qids,semantic_labels
0,Q31,en,Belgium,"Belgium, officially the Kingdom of Belgium, is...","['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '..."
1,Q31,de,Belgien,"Belgien (amtlich Königreich Belgien, niederlän...","['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '..."
2,Q31,es,Bélgica,Bélgica (en neerlandés: ; en francés: ; en ale...,"['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '..."
3,Q31,zh,比利时,比利時王國（法語：Royaume de Belgique；荷蘭語：Koninkrijk Be...,"['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '..."
4,Q31,hi,बेल्जियम,"बेल्जियम (डच: België, फ़्रांसीसी: Belgique, जर...","['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '..."
5,Q8,en,Happiness,Happiness is a complex and multifaceted emotio...,"['Q331769', 'Q60539479']","['mood', 'positive emotion']"
6,Q8,de,Glück,"Glück ist ein mehrdeutiger Begriff, der moment...","['Q331769', 'Q60539479']","['mood', 'positive emotion']"
7,Q8,es,Felicidad,La felicidad es una emoción o estado de ánimo ...,"['Q331769', 'Q60539479']","['mood', 'positive emotion']"
8,Q8,zh,幸福,幸福（古希臘語：εὐδαιμονία、拉丁語：felicitas、英語：felicity、h...,"['Q331769', 'Q60539479']","['mood', 'positive emotion']"
9,Q8,hi,प्रसन्नता,प्रसन्नता मानवों में पाई जाने वाली भावनाओं में...,"['Q331769', 'Q60539479']","['mood', 'positive emotion']"


In [39]:
import os
import pandas as pd
import requests

# ------------------ DOWNLOADABLE LEXICONS ------------------

# Ensure lexicons directory exists
os.makedirs("lexicons", exist_ok=True)

def download_file(url, path):
    if not os.path.exists(path):
        print(f"Downloading {path} ...")
        r = requests.get(url)
        r.raise_for_status()
        with open(path, 'wb') as f:
            f.write(r.content)

# English – NRC Emotion Lexicon
download_file(
    "https://raw.githubusercontent.com/aditeyabaral/lok-sabha-election-twitter-analysis/master/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt",
    "lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"
)

# Spanish – NRC (Community version)
download_file(
    "https://raw.githubusercontent.com/jboscomendoza/lexicos-nrc-afinn/master/lexico_nrc.csv",
    "lexicons/nrc_spanish.csv"
)

# Chinese – NTUSD
download_file(
    "https://raw.githubusercontent.com/sweslo17/chinese_sentiment/master/dict/ntusd-positive.txt",
    "lexicons/ntusd-positive.txt"
)
download_file(
    "https://raw.githubusercontent.com/sweslo17/chinese_sentiment/master/dict/ntusd-negative.txt",
    "lexicons/ntusd-negative.txt"
)

# German – SentiWS
download_file(
    "https://raw.githubusercontent.com/vani-gupta/Twitter-Sentiment-Analysis/main/SentiWS_v2.0_Positive.txt",
    "lexicons/SentiWS_v2.0_Positive.txt"
)
download_file(
    "https://raw.githubusercontent.com/vani-gupta/Twitter-Sentiment-Analysis/main/SentiWS_v2.0_Negative.txt",
    "lexicons/SentiWS_v2.0_Negative.txt"
)

# Hindi – Affectivespace
download_file(
    "https://raw.githubusercontent.com/vani-gupta/Twitter-Sentiment-Analysis/main/affectivespace.xlsx",
    "lexicons/affectivespace.xlsx"
)

# ------------------ LEXICON LOADERS ------------------

def load_nrc_english(filepath):
    df = pd.read_csv(filepath, sep='\t', names=['word', 'emotion', 'association'])
    pos = df[(df['emotion'] == 'positive') & (df['association'] == 1)]['word']
    neg = df[(df['emotion'] == 'negative') & (df['association'] == 1)]['word']
    return set(pos), set(neg)

def load_nrc_spanish(filepath):
    df = pd.read_csv(filepath)
    pos = df[df['sentimiento'] == 'positivo']['palabra']
    neg = df[df['sentimiento'] == 'negativo']['palabra']
    return set(pos), set(neg)

def load_ntusd(pos_path, neg_path):
    pos = pd.read_csv(pos_path, header=None)[0]
    neg = pd.read_csv(neg_path, header=None)[0]
    return set(pos), set(neg)

def load_sentiws(pos_path, neg_path):
    def read_sentiws(path):
        with open(path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        words = {}
        for line in lines:
            parts = line.strip().split('\t')
            word = parts[0].split('|')[0]
            score = float(parts[1])
            words[word] = score
            if len(parts) > 2:
                for infl in parts[2].split(','):
                    words[infl] = score
        return words

    pos_dict = read_sentiws(pos_path)
    neg_dict = read_sentiws(neg_path)
    return {**pos_dict, **neg_dict}

def load_hindi_swn(filepath):
    df = pd.read_excel(filepath, usecols=[0, 1], names=["word", "score"], header=None)
    return df.set_index("word")["score"].to_dict()

# ------------------ LOAD ALL LEXICONS ------------------

lexicons = {
    'en': load_nrc_english("lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"),
    'es': load_nrc_spanish("lexicons/nrc_spanish.csv"),
    'zh': load_ntusd("lexicons/ntusd-positive.txt", "lexicons/ntusd-negative.txt"),
    'de': load_sentiws("lexicons/SentiWS_v2.0_Positive.txt", "lexicons/SentiWS_v2.0_Negative.txt"),
    'hi': load_hindi_swn("lexicons/affectivespace.xlsx")
}


In [40]:
import jieba

# ------------------ SCORING FUNCTION ------------------

def score_text(text, lang):
    words = str(text).split()
    if lang == 'hi' or lang == 'de':
        senti = lexicons[lang]
        scores = [senti.get(w, 0) for w in words if w in senti]
        return sum(scores) / len(scores) if scores else 0
    elif lang == 'zh':  # For Mandarin Chinese
        if not isinstance(text, str):
            text = str(text) if not pd.isna(text) else ''
        words = jieba.lcut(text)  # Use Jieba to tokenize the Chinese text
        pos_set, neg_set = lexicons['zh']
        scores = [1 if w in pos_set else -1 if w in neg_set else 0 for w in words]
        return sum(scores) / len(words) if words else 0
    else:
        pos, neg = lexicons[lang]
        score = sum(1 for w in words if w in pos) - sum(1 for w in words if w in neg)
        return score / len(words) if words else 0

In [41]:
# ------------- PROCESS CSV AND ADD SCORES -------------
df_articles['sentiment_score'] = df_articles.apply(lambda row: score_text(row['extract'], row['language']), axis=1)

# ------------- SAVE TO CSV -------------
df_articles.to_csv("wiki_extracts_scored.csv", index=False)
print("Sentiment scores added and saved to 'wiki_extracts_scored.csv'")

Sentiment scores added and saved to 'wiki_extracts_scored.csv'


In [42]:
df_articles.head()

,qid,language,title,extract,semantic_qids,semantic_labels,sentiment_score
0,Q31,en,Belgium,"Belgium, officially the Kingdom of Belgium, is...","['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '...",0.020315
1,Q31,de,Belgien,"Belgien (amtlich Königreich Belgien, niederlän...","['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '...",0.020433
2,Q31,es,Bélgica,Bélgica (en neerlandés: ; en francés: ; en ale...,"['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '...",0.018063
3,Q31,zh,比利时,比利時王國（法語：Royaume de Belgique；荷蘭語：Koninkrijk Be...,"['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '...",0.015078
4,Q31,hi,बेल्जियम,"बेल्जियम (डच: België, फ़्रांसीसी: Belgique, जर...","['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '...",0.069541


In [43]:
df_articles.shape

(5000, 7)

In [44]:
df_articles[df_articles["language"] == "zh"].head()

,qid,language,title,extract,semantic_qids,semantic_labels,sentiment_score
3,Q31,zh,比利时,比利時王國（法語：Royaume de Belgique；荷蘭語：Koninkrijk Be...,"['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '...",0.015078
8,Q8,zh,幸福,幸福（古希臘語：εὐδαιμονία、拉丁語：felicitas、英語：felicity、h...,"['Q331769', 'Q60539479']","['mood', 'positive emotion']",0.024049
13,Q23,zh,乔治·华盛顿,乔治·华盛顿（英語：George Washington，1732年2月22日—1799年12...,['Q5'],['human'],0.000932
18,Q2013,zh,维基数据,維基數據（英語：Wikidata）是一個可協同編輯的知識庫，是繼2006年的維基學院之後，第...,"['Q33120876', 'Q638153', 'Q36509592', 'Q156335...","['Wikimedia content project', 'semantic wiki',...",0.003077
23,Q45,zh,葡萄牙,葡萄牙共和國（葡萄牙語：República Portuguesa），簡稱葡萄牙或葡國（Por...,"['Q3624078', 'Q6256', 'Q20181813']","['sovereign state', 'country', 'colonial power']",0.006471


In [45]:
df_articles[df_articles["language"] == "hi"].head()

,qid,language,title,extract,semantic_qids,semantic_labels,sentiment_score
4,Q31,hi,बेल्जियम,"बेल्जियम (डच: België, फ़्रांसीसी: Belgique, जर...","['Q3624078', 'Q43702', 'Q6256', 'Q20181813', '...","['sovereign state', 'federation', 'country', '...",0.069541
9,Q8,hi,प्रसन्नता,प्रसन्नता मानवों में पाई जाने वाली भावनाओं में...,"['Q331769', 'Q60539479']","['mood', 'positive emotion']",0.067310
14,Q23,hi,जॉर्ज वॉशिंगटन,जार्ज वाशिंगटन (अंग्रेज़ी: George Washington) ...,['Q5'],['human'],0.094267
19,Q2013,hi,विकिडेटा,विकिडेटा अथवा विकिडाटा (अंग्रेज़ी: Wikidata) व...,"['Q33120876', 'Q638153', 'Q36509592', 'Q156335...","['Wikimedia content project', 'semantic wiki',...",-0.050336
24,Q45,hi,पुर्तगाल,पुर्तगाली गणराज्य यूरोप खंड में स्थित देश है। ...,"['Q3624078', 'Q6256', 'Q20181813']","['sovereign state', 'country', 'colonial power']",0.043690


In [46]:
df_articles[df_articles["qid"] == "Q102"].head()

,qid,language,title,extract,semantic_qids,semantic_labels,sentiment_score
45,Q102,en,Pneumonoultramicroscopicsilicovolcanoconiosis,Pneumono­ultra­micro­scopic­silico­volcano­con...,['Q101991'],['inflammation'],0.020501
46,Q102,de,Pneumonoultramicroscopicsilicovolcanoconiosis,Pneumonoultramicroscopicsilicovolcanoconiosis ...,['Q101991'],['inflammation'],-0.178280
47,Q102,es,Pneumonoultramicroscopicsilicovolcanoconiosis,Pneumonoultramicroscopicsilicovolcanoconosis e...,['Q101991'],['inflammation'],0.008475
48,Q102,zh,火山肺矽病,Pneumonoultramicroscopicsilicovolcanoconiosis（...,['Q101991'],['inflammation'],0.000000
49,Q102,hi,सिलिकामय धूलि फुप्फुसार्ति,सिलिकामय धूलि फुप्फुसार्ति एक व्यावसायिक श्वसन...,['Q101991'],['inflammation'],-0.140082


In [60]:
df_articles[(df_articles["sentiment_score"] < 0) & (df_articles["language"] == "de")].shape

(605, 7)

In [61]:
df_articles[(df_articles["sentiment_score"] < 0) & (df_articles["language"] == "en")].shape

(44, 7)

In [62]:
df_articles[(df_articles["sentiment_score"] < 0) & (df_articles["language"] == "hi")].shape

(433, 7)

In [51]:
df_articles[(df_articles["sentiment_score"] < 0) & (df_articles["language"] == "zh")].shape

(322, 7)

In [52]:
df_articles[(df_articles["sentiment_score"] < 0) & (df_articles["language"] == "es")].shape

(75, 7)

In [63]:
df_articles[(df_articles["sentiment_score"] == 0)].shape

(248, 7)

In [64]:
df_articles[(df_articles["sentiment_score"] == 0)].head(20)

,qid,language,title,extract,semantic_qids,semantic_labels,sentiment_score
48,Q102,zh,火山肺矽病,Pneumonoultramicroscopicsilicovolcanoconiosis（...,['Q101991'],['inflammation'],0.0
62,Q144,es,Perro,NaN,['Q55983715'],['organisms known by a particular common name'],0.0
97,Q210,es,Ángulo recto,Un ángulo recto es aquel que mide 90° (sexages...,[],[],0.0
138,Q411,zh,天體生物學,天體生物學（英語：astrobiology），舊稱外空生物学（xenobiology），是一...,"['Q28598684', 'Q131565179', 'Q11862829', 'Q104...","['branch of biology', 'branch of astronomy', '...",0.0
179,Q550,hi,शॉन्ज़-एलिसीज़,Champs-Élysées (फ़्रांसीसी pronunciation: [ʃɑ̃...,"['Q7543083', 'Q570116', 'Q54114']","['avenue', 'tourist attraction', 'boulevard']",0.0
228,Q659,zh,北,北是相對於南的方位名，作爲方位角的起點。面向東方時，北就在左邊。有“上北，下南，左西，右东”...,"['Q23718', 'Q11114344']","['cardinal direction', 'points of the compass']",0.0
278,Q853,zh,安德烈·塔尔科夫斯基,安德烈·阿尔谢尼耶维奇·塔尔科夫斯基（俄语：Андре́й Арсе́ньевич Тарк...,['Q5'],['human'],0.0
323,Q901,zh,科学家,科学家是一个泛称，广义上指使用系统化的活动来发现新知识的人。狭义的定义指使用科学方法做研究，...,"['Q28640', 'Q4164871']","['profession', 'position']",0.0
404,Q1201,hi,सारलैंड,NaN,"['Q1221156', 'Q47574']","['federated state of Germany', 'unit of measur...",0.0
436,Q1281,de,Kategorie:!Hauptkategorie,Diese Kategorie bildet den Einstiegspunkt (Sta...,"['Q15647814', 'Q4167836']","['Wikimedia administration category', 'Wikimed...",0.0


In [65]:
df_articles[df_articles["qid"] == "Q3569"].head()

,qid,language,title,extract,semantic_qids,semantic_labels,sentiment_score
1045,Q3569,en,Senegal River,"The Senegal River (Serer: ""Seen O Gal"" or ""Sen...",['Q4022'],['river'],0.010996
1046,Q3569,de,Senegal (Fluss),Der Senegal ist ein 1086 Kilometer langer Stro...,['Q4022'],['river'],0.046248
1047,Q3569,es,Río Senegal,El río Senegal es un río de África Occidental ...,['Q4022'],['river'],0.015921
1048,Q3569,zh,塞内加尔河,塞内加尔河（法語：Fleuve Sénégal、阿拉伯语：سينغال）位于非洲西部，由巴科...,['Q4022'],['river'],0.000000
1049,Q3569,hi,सेनेगल नदी,"सेनेगल नदी पश्चिम अफ्रीका में 1,086 कि॰मी॰ (67...",['Q4022'],['river'],0.202258
